# Build micrograd: a hands-on notebook

A student workspace for following the first Neural Networks: Zero to Hero lesson. Work through the prompts in order and fill in the marked exercises yourself; this notebook is a scaffold, not a completed implementation.

The goal is to build a tiny scalar-valued reverse-mode autodiff engine, then use it to train a small multilayer perceptron. No packages beyond Python are required. Run cells from top to bottom, rerunning earlier definitions as you improve them.

## 1. A scalar that remembers its computation

A `Value` will hold a number and, later, the information needed to differentiate a result with respect to its inputs. Start by implementing construction and a useful representation. Then add arithmetic operators one at a time. Each operation should return a new `Value` and remember its parent values and operation.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op="", label=""):
        # TODO: store data, grad (initially 0), label, operation, and parents.
        # Each value should also have a _backward function that initially does nothing.
        raise NotImplementedError("Implement Value.__init__")

    def __repr__(self):
        # TODO: return a short representation that includes the data.
        raise NotImplementedError("Implement Value.__repr__")

    def __add__(self, other):
        # TODO: support adding another Value or a Python number.
        # Save a closure on the result that accumulates its gradient into both inputs.
        raise NotImplementedError("Implement Value.__add__")

    def __mul__(self, other):
        # TODO: support multiplying another Value or a Python number.
        # Use the product rule in the result's _backward closure.
        raise NotImplementedError("Implement Value.__mul__")

    def __neg__(self):
        # TODO: implement negation using an operation you already have.
        raise NotImplementedError("Implement Value.__neg__")

    def __sub__(self, other):
        # TODO: implement subtraction using addition and negation.
        raise NotImplementedError("Implement Value.__sub__")

    def __pow__(self, exponent):
        # TODO: support a numeric exponent and define the local derivative.
        raise NotImplementedError("Implement Value.__pow__")

    def __truediv__(self, other):
        # TODO: implement division using a negative power.
        raise NotImplementedError("Implement Value.__truediv__")

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __rsub__(self, other):
        return (-self) + other

    def __rtruediv__(self, other):
        return self ** -1 * other

    def tanh(self):
        # TODO: compute tanh(self.data), then save its local derivative.
        raise NotImplementedError("Implement Value.tanh")

    def exp(self):
        # TODO: compute exp(self.data), then save its local derivative.
        raise NotImplementedError("Implement Value.exp")

    def relu(self):
        # TODO: compute max(0, self.data) and save the local derivative.
        raise NotImplementedError("Implement Value.relu")

    def backward(self):
        # TODO: visit each node once in topological order, seed this node's
        # gradient with 1, then call each node's _backward in reverse order.
        raise NotImplementedError("Implement Value.backward")

## 2. Check the arithmetic and gradients

Use these small checks as you implement each operation. For the final gradient check, compare autodiff with a finite-difference estimate. A repeated edge (for example, `a * a`) is a useful reminder that gradients must be **accumulated**, not overwritten.

In [ ]:
a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = a * b + 4
assert c.data == -2.0

x = Value(1.5)
y = (x * x + 2 * x + 1).tanh()
y.backward()

# TODO: check y.grad is 1 and compare x.grad to a centered finite difference.
# Try changing x and the expression to test your implementation.

## 3. Build a neuron and a multilayer perceptron

A neuron computes a weighted sum plus a bias, followed by `tanh`. A layer is a list of neurons; an MLP is a list of layers. Initialize weights to small random values and biases to zero. Keep every parameter as a `Value` so gradients flow through the whole model.

In [ ]:
import random

class Neuron:
    def __init__(self, nin):
        # TODO: create nin small random Value weights and one zero bias.
        raise NotImplementedError("Implement Neuron.__init__")

    def __call__(self, x):
        # TODO: return tanh(sum(w_i * x_i) + bias).
        raise NotImplementedError("Implement Neuron.__call__")

    def parameters(self):
        # TODO: return this neuron's weights and bias.
        raise NotImplementedError("Implement Neuron.parameters")

class Layer:
    def __init__(self, nin, nout):
        # TODO: create nout neurons, each with nin inputs.
        raise NotImplementedError("Implement Layer.__init__")

    def __call__(self, x):
        # TODO: apply each neuron; return a list of outputs.
        raise NotImplementedError("Implement Layer.__call__")

    def parameters(self):
        # TODO: flatten the parameters from all neurons in this layer.
        raise NotImplementedError("Implement Layer.parameters")

class MLP:
    def __init__(self, nin, nouts):
        # TODO: create layers for each adjacent pair in [nin] + nouts.
        raise NotImplementedError("Implement MLP.__init__")

    def __call__(self, x):
        # TODO: pass x through each layer in sequence.
        raise NotImplementedError("Implement MLP.__call__")

    def parameters(self):
        # TODO: collect parameters from all layers.
        raise NotImplementedError("Implement MLP.parameters")

In [ ]:
# A tiny four-point classification dataset (labels are -1 or +1).
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

model = MLP(3, [4, 4, 1])
predictions = [model(x) for x in xs]
# TODO: inspect the predictions and verify there are four scalar outputs.

## 4. Define a loss and train

Use a simple max-margin loss: for each example, penalize predictions that do not have the correct sign with enough margin. Add a small squared-parameter penalty to discourage unnecessarily large weights. Then repeat the training steps: clear parameter gradients, run a forward pass, compute the loss, backpropagate, and update parameters by a small step opposite the gradient.

In [ ]:
def loss_for(model, xs, ys):
    # TODO: get one scalar prediction per example (the model has one output).
    # For each pair (score, label), use (1 - score * label).relu().
    # Add a small L2 penalty over model.parameters() and return one Value.
    raise NotImplementedError("Implement loss_for")

In [ ]:
parameters = model.parameters()
for step in range(100):
    # TODO: set every parameter's grad to 0 before each backward pass.
    loss = loss_for(model, xs, ys)
    loss.backward()

    learning_rate = 0.05
    for parameter in parameters:
        # TODO: update parameter.data using its gradient and learning_rate.
        pass

    if step % 10 == 0:
        print(f"step {step:3d} | loss {loss.data:.4f}")

# TODO: print predictions after training and check their signs against ys.

## 5. Experiments

- Try a different network shape or learning rate. What changes?
- Remove the regularization term. How does training behave?
- Add labels to intermediate `Value`s and sketch the computation graph on paper.
- Compare a gradient with a finite-difference estimate at several parameter values.

When you are done, restart the kernel and run all cells from the top to make sure the notebook works as a complete lesson.